In [ ]:
#code or drive mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/StanfordDogs/stanford_dogs_cleaned.zip'   # update if path differs
extract_path = '/content/cleaned_dataset'

# Unzip only if not already extracted
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset unzipped successfully!")
else:
    print("✅ Dataset already extracted.")


✅ Dataset unzipped successfully!


In [ ]:
import os
import shutil

def clean_dataset_folder(root_dir):
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')
    removed_dirs = 0
    removed_files = 0

    for root, dirs, files in os.walk(root_dir):
        # Remove .ipynb_checkpoints folders
        for d in dirs:
            if d == ".ipynb_checkpoints":
                full_path = os.path.join(root, d)
                shutil.rmtree(full_path)
                removed_dirs += 1

        # Remove non-image files
        for f in files:
            if not f.lower().endswith(valid_extensions):
                full_path = os.path.join(root, f)
                os.remove(full_path)
                removed_files += 1

    print(f"✅ Cleanup complete: {removed_dirs} folders and {removed_files} files removed.")

# 🔧 Run this on your dataset folder
clean_dataset_folder("/content/cleaned_dataset")

✅ Cleanup complete: 0 folders and 1 files removed.


In [ ]:
from sklearn.model_selection import train_test_split
import shutil

def split_train_valid(clean_dir, train_dir, valid_dir, test_size=0.2):
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(valid_dir, exist_ok=True)

    breeds = [b for b in os.listdir(clean_dir) if os.path.isdir(os.path.join(clean_dir, b))]

    for breed in breeds:
        breed_path = os.path.join(clean_dir, breed)
        images = [f for f in os.listdir(breed_path) if f.lower().endswith(('.jpg', '.png'))]

        train_imgs, valid_imgs = train_test_split(images, test_size=test_size, random_state=42)

        os.makedirs(os.path.join(train_dir, breed), exist_ok=True)
        os.makedirs(os.path.join(valid_dir, breed), exist_ok=True)

        for img in train_imgs:
            shutil.copy(os.path.join(breed_path, img), os.path.join(train_dir, breed, img))
        for img in valid_imgs:
            shutil.copy(os.path.join(breed_path, img), os.path.join(valid_dir, breed, img))

    print("✅ Dataset split complete! Train and Validation folders are ready.")



In [ ]:
split_train_valid("/content/cleaned_dataset", "/content/train", "/content/valid")


✅ Dataset split complete! Train and Validation folders are ready.


In [ ]:
import os
import json
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ==============================
# 1️⃣ Paths
# ==============================
train_dir = "/content/train"
val_dir = "/content/valid"
label_map_path = "label_map.json"
model_save_path = "dog_breed_mobilenetv2_finetuned.keras"

# ==============================
# 2️⃣ Data Generators
# ==============================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=(224, 224), batch_size=32, class_mode='categorical'
)

val_gen = val_datagen.flow_from_directory(
    val_dir, target_size=(224, 224), batch_size=32, class_mode='categorical'
)

num_classes = train_gen.num_classes
print(f"✅ Detected {num_classes} classes.")

# ==============================
# 3️⃣ Save Label Map
# ==============================
label_map = {str(v): k for k, v in train_gen.class_indices.items()}
with open(label_map_path, "w") as f:
    json.dump(label_map, f)
print("✅ label_map.json saved.")

# ==============================
# 4️⃣ Build Model (Functional API)
# ==============================
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ==============================
# 5️⃣ Callbacks
# ==============================
checkpoint = ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)

# ==============================
# 6️⃣ Train for 10 Epochs
# ==============================
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[checkpoint, early_stop],
    verbose=1
)

# ==============================
# 7️⃣ Evaluate + Save
# ==============================
loss, acc = model.evaluate(val_gen)
print(f"\n✅ Final Validation Accuracy: {acc * 100:.2f}%")
print(f"✅ Final Validation Loss: {loss:.4f}")

model.save(model_save_path)
print("💾 Final model saved as .keras format.")

Found 10728 images belonging to 120 classes.
Found 2742 images belonging to 120 classes.
✅ Detected 120 classes.
✅ label_map.json saved.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 120)            │        30,840 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,616,760 (9.98 MB)

 Trainable params: 358,776 (1.37 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
336/336 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.0274 - loss: 4.9232
Epoch 1: val_accuracy improved from -inf to 0.26404, saving model to best_model.keras
336/336 ━━━━━━━━━━━━━━━━━━━━ 763s 2s/step - accuracy: 0.0274 - loss: 4.9223 - val_accuracy: 0.2640 - val_loss: 3.8502
Epoch 2/10
336/336 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1418 - loss: 3.9086
Epoch 2: val_accuracy improved from 0.26404 to 0.48468, saving model to best_model.keras
336/336 ━━━━━━━━━━━━━━━━━━━━ 770s 2s/step - accuracy: 0.1420 - loss: 3.9078 - val_accuracy: 0.4847 - val_loss: 2.5106
Epoch 3/10
336/336 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2966 - loss: 2.9433
Epoch 3: val_accuracy improved from 0.48468 to 0.60102, saving model to best_model.keras
336/336 ━━━━━━━━━━━━━━━━━━━━ 773s 2s/step - accuracy: 0.2966 - loss: 2.9429 - val_accuracy: 0.6010 - val_loss: 1.7802
Epoch 4/10
336/336 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3726 - loss: 2.4422
Epoch 4: val_accuracy improved from 0.6010